<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/8_Low_level_API_%2B_quantization_%2B_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this notebook, we explore Hugging Face's low-level Transformers APIs for running and comparing open-source instruction-tuned LLMs such as Llama 3.1, Phi-3.5, Gemma 2 and Qwen 2.

We load models and tokenizers using AutoModelForCausalLM and AutoTokenizer, apply chat templates, tokenize prompts, and generate responses using the generate() API.

To fit larger models on a Colab L4 GPU, we used 4-bit quantization with BitsAndBytes (nf4 quantization).



# Needed Libraries

1. **requests** :
    * **Purpose:** Simple HTTP library for making API requests.
    * **Use case:** Useful for downloading models, datasets, or interacting with REST APIs.

2. **torch**
    * **Purpose:** PyTorch, a deep learning framework developed by Facebook.
    * **Use case:** For building, training, and running neural networks.

3. **bitsandbytes**
    * **Purpose:** A lightweight CUDA extension for 8-bit and 4-bit optimizers and matrix multiplication.
    * **Use case:** Used to reduce memory usage and speed up large models, especially in inference or fine-tuning. Commonly used with Hugging Face models.

4. **transformers**
    * **Purpose:** Hugging Face's Transformers library.
    * **Use case:** Provides pre-trained transformer models like BERT, GPT, T5, etc., with easy APIs for text generation, classification, etc.

5. **sentencepiece**
    * **Purpose:** Tokenizer developed by Google for unsupervised text tokenization.
    * **Use case:** Many Hugging Face models (e.g., T5, mBART) use it for handling subword units.

6. **accelerate**
    * **Purpose:** Another Hugging Face library for optimizing model training and inference.
    * **Use case:** Helps scale training across CPUs, GPUs, or even TPUs with minimal code changes. Works great in multi-GPU setups too.

In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate

#-q means quiet mode, so it suppresses the usual output during installation—keeps the notebook or terminal cleaner.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import torch
import gc
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig
)

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Models

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"   # Meta
PHI35 = "microsoft/Phi-3.5-mini-instruct"         # Microsoft
GEMMA2 = "google/gemma-2-2b-it"                   # Google
QWEN2 = "Qwen/Qwen2-7B-Instruct"                  # Alibaba

In [ ]:
def form_message(prompt):

  message = [
      {"role": "system", "content": "You are a helpful assistant"},
      {"role": "user", "content": prompt}
     ]
  return message


# Quantization

In machine learning — especially deep learning — quantization is the process of reducing the precision of the numbers used to represent a model's parameters (like weights and activations).

Normally, models are trained and stored using 32-bit floating-point numbers (float32).
Quantization reduces this to lower precision types like:
  1. 16-bit (e.g., float16, bfloat16)
  2. 8-bit (e.g., int8)
  3. 4-bit (e.g., nf4, fp4)

**Why Quantize?**
Quantization is mainly used for efficiency:

  * Lower Memory	- Model takes up less RAM/VRAM
  * Faster Inference	- Smaller numbers → faster computation on some hardware
  * Lower Power - Especially useful for edge devices (phones, Raspberry Pi, etc.)



In [ ]:
#Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
#GPU Memory Utility

def show_gpu_memory():

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"Allocated : {allocated:.2f} GB")
    print(f"Reserved  : {reserved:.2f} GB")

In [ ]:
show_gpu_memory()

In [ ]:
#Load Model - Load once, use many times.

def load_model(model_name):

    print(f"\nLoading {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config #,
        #trust_remote_code=True
    )

    model.config.pad_token_id = tokenizer.pad_token_id

    footprint = model.get_memory_footprint() / 1024**3

    print(f"Model footprint: {footprint:.2f} GB")

    show_gpu_memory()

    return model, tokenizer

In [ ]:
#Generate - This is model-agnostic

def generate(
    model,
    tokenizer,
    messages,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    stream=True
):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    device = next(model.parameters()).device

    inputs = {k: v.to(device) for k, v in inputs.items()}

    streamer = None

    if stream:
        streamer = TextStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            streamer=streamer
        )

    response = tokenizer.decode(
        outputs[0].cpu(),
        skip_special_tokens=True
    )

    del inputs
    del outputs

    if streamer is not None:
        del streamer

    gc.collect()
    torch.cuda.empty_cache()

    return response

In [ ]:
#Cleanup functions : since we want to use multiple models

def cleanup():

    gc.collect()

    torch.cuda.empty_cache()

    print(
        f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
    )

    print(
        f"Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB"
    )

############################################################################

def count_cuda_tensors():

    import gc

    count = 0

    for obj in gc.get_objects():

        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                count += 1
        except:
            pass

    print(f"CUDA tensors alive: {count}")

############################################################################

def hard_cleanup():

    gc.collect()

    torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(
        f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
    )

    print(
        f"Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB"
    )

############################################################################

def switch_model(model, tokenizer):

    del model
    del tokenizer

    gc.collect()
    torch.cuda.empty_cache()

    show_gpu_memory()

    count_cuda_tensors()

In [ ]:
# let's try the first model
model, tokenizer = load_model(LLAMA)

In [ ]:
#question
message = form_message("What is the capital of Japan ?")

response = generate(
    model,
    tokenizer,
    message,
    max_new_tokens=100
)

In [ ]:
#question
message = form_message("Explain evaporation is simple terms ?")

response = generate(
    model,
    tokenizer,
    message,
    max_new_tokens=100
)

In [ ]:
#clean up before we switch models
print("Show GPU memory before clean-up")
show_gpu_memory()
print("Show GPU memory after clean-up")
del model
del tokenizer
cleanup()
print("tensors remaining after clean-up")
count_cuda_tensors()

In [ ]:
#Load next model
model, tokenizer = load_model(PHI35)

In [ ]:
message = form_message("In which continent is India ?")

response = generate(
    model,
    tokenizer,
    message,
    max_new_tokens=100
)


In [ ]:
message = form_message("Tell me a joke about engineers ?")

response = generate(
    model,
    tokenizer,
    message,
    max_new_tokens=100
)

In [ ]:
#clean up before we switch models
print("Show GPU memory before clean-up")
show_gpu_memory()
print("Show GPU memory after clean-up")
del model
del tokenizer
cleanup()
print("tensors remaining after clean-up")
count_cuda_tensors()

**Gemma from Google requires us to accept their terms in Hugging Face.**

  * Visit this page to ask for access -
    https://huggingface.co/google/gemma-2-2b-it

In [ ]:
model, tokenizer = load_model(GEMMA2)

In [ ]:
message_gemma = [{"role": "user", "content": "Tell a light-hearted joke for a room of Doctors"}]
# since Gemma from Google does not support system role

response = generate(
    model,
    tokenizer,
    message_gemma,
    max_new_tokens=100
)


In [ ]:
message_gemma = [{"role": "user", "content": "What is the capital of Botswana ?"}]
# since Gemma from Google does not support system role

response = generate(
    model,
    tokenizer,
    message_gemma,
    max_new_tokens=100
)


In [ ]:
#clean up before we switch models
print("Show GPU memory before clean-up")
show_gpu_memory()
print("Show GPU memory after clean-up")
del model
del tokenizer
cleanup()
print("tensors remaining after clean-up")
count_cuda_tensors()

In [ ]:
# Try different model
model, tokenizer = load_model(QWEN2)

In [ ]:
message = form_message(" What sport does Sachin Tendulkar play ?")

response = generate(
    model,
    tokenizer,
    message,
    max_new_tokens=100
)
